# HEART v2 | General-ligand integrity + H200 10,000-molecule evaluation

**Two independent, read-only scientific questions:**

**Part I — ligand provenance / deduplication:** What fraction of the actual general `train.lmdb` records share a molecular identity? How many distinct molecules are present, and how much train–valid overlap is there? Compare the result against **LUMI-lab's reported 13,369,320 distinct general molecules**, *without attempting to force the count to 13M*.

**Part II — learned representations / original training objectives:** On a reproducible, fixed sample of **10,000** `valid.lmdb` molecules, load the **completed H200 `checkpoint_best.pt`** and evaluate masked atom prediction, geometry recovery (coordinate and distance), NT-Xent contrastive objective, and CLS/projected embeddings (PCA/UMAP). For the trained model, use the same HEART task, collater and `unimol_contrastive` loss that produced the checkpoint.

This notebook is **standalone** (no external audit/evaluation helper scripts), never calls `sbatch`, never starts continual training, never changes the `.lmdb` or `.pt`, and never installs packages. Outputs are created in a separate `/scratch/.../heart_evaluation/` folder. The attached historic 192-molecule notebook is a *style and implementation reference*, **not a file dependency**.

**Important:** A full exact scan of 18.8M records is substantial CPU/storage I/O; it must read every LMDB record. It can be run/resumed directly from Jupyter. `RUN_FULL_RAW_AUDIT` defaults to `False` so opening the notebook cannot accidentally begin a multi-hour scan. Set it to `True` when you are ready to obtain the exact result. Canonical RDKit deduplication is separately opt-in and much more expensive.


## A — HPC paths, kernel and controls

Paths follow the **actual moved `/scratch` layout**. The v2 repository and `UniMolReference` are siblings; do not import from the historical checkout. The checkpoint is read from v2 if present. The notebook refuses to silently use another checkpoint. Select the existing `heart_gpu` kernel in Jupyter; terminal `conda activate` alone does not change the Jupyter kernel.


In [ ]:
from pathlib import Path
import os, sys, json, time, math, platform, importlib, hashlib, pickle, sqlite3, gc
import numpy as np
from IPython.display import display, Markdown

BASE = Path('/scratch/ypan/heart_liver/model_train')
REPO_V2 = BASE / 'heart-molecular-pretraining-v2'
MODEL_ROOT = REPO_V2 / 'model'
DATA_DIR = BASE / 'UniMolReference' / 'ligand_data' / 'ligands'
UNICORE_ROOT = BASE / 'third_party' / 'Uni-Core'
CHECKPOINT = REPO_V2 / 'runs/h200/full/heart-h200-b448-v1/checkpoint_best.pt'
OUTPUT_ROOT = BASE / 'heart_evaluation' / 'general_ligand_audit_and_10k'

# I/O: explicitly opt into the full scan; preview is NOT an exact dataset-wide duplicate rate.
RUN_PREVIEW = True
PREVIEW_RECORDS_PER_SPLIT = 20000
RUN_FULL_RAW_AUDIT = False      # Set True for exact full LMDB scan (can take hours).
RUN_FULL_CANONICAL_AUDIT = False # Optional: full RDKit canonical-isomeric-SMILES audit (slower).

# Evaluation: keep original H200 notebook's two-view collater and batch of eight molecules.
RUN_EVALUATION = False          # Set True to evaluate all 10,000 directly in this Jupyter kernel.
N_EVAL = 10000
BATCH_MOLECULES = 8            # original 192-molecule notebook used 8 (two views = 16 rows)
SAMPLE_SEED = 17
VALID_EPOCH = 1
DEVICE_PREFERENCE = 'auto'     # 'auto', 'cpu', or 'cuda'; no Slurm submitted
TORCH_THREADS = 4

for label, path in [('v2 repository',REPO_V2),('v2 model',MODEL_ROOT),('ligand train',DATA_DIR/'train.lmdb'),
                    ('ligand valid',DATA_DIR/'valid.lmdb'),('ligand dict',DATA_DIR/'dict.txt'),
                    ('H200 best checkpoint',CHECKPOINT),('Uni-Core',UNICORE_ROOT)]:
    print(f'{label:25s} {"OK" if path.exists() else "MISSING":8s} {path}')
required = [MODEL_ROOT/'unimol'/'tasks'/'unimol_contrastive.py',DATA_DIR/'train.lmdb', DATA_DIR/'valid.lmdb',
            DATA_DIR/'dict.txt',CHECKPOINT,UNICORE_ROOT/'unicore']
missing = [p for p in required if not p.is_file() and not p.is_dir()]
if missing: raise FileNotFoundError('Correct path configuration first: '+', '.join(map(str,missing)))
print('Python kernel:',sys.executable, '| prefix:',sys.prefix)
print('Expected HEART environment contains heart_gpu:', 'heart_gpu' in str(sys.prefix))
if 'heart_gpu' not in str(sys.prefix):
    print('WARNING: choose the heart_gpu Jupyter kernel before Part II. Part I needs lmdb; Part II needs torch/unicore.')
assert not OUTPUT_ROOT.resolve().is_relative_to(DATA_DIR.resolve())
assert not OUTPUT_ROOT.resolve().is_relative_to(REPO_V2/'runs/h200')


## B — Exact meaning of duplicate

- **Raw identity:** exactly matching original `record['smi']` text; this is a reproducible, lossless first audit, but equivalent SMILES may use different atom ordering.
- **Canonical identity:** RDKit `Chem.MolToSmiles(Chem.MolFromSmiles(smi), isomericSmiles=True)`; this groups some equivalent molecular representations (stereochemistry preserved where specified). It is **not** InChI-standardized salt/tautomer normalization, and may differ from LUMI's unspecified preprocessing.
- One LMDB **record may contain multiple 3D conformations**. The audit counts molecular identities from `smi`, never treats each conformer as a new molecule.
- **Train–valid overlap** is computed at the chosen SMILES identity level. A good loss curve is not sufficient to rule out overlap.
- `18,837,320` is a *historical count*; the code below reads the actual LMDB stats, not the previous log.


In [ ]:
import lmdb

def open_lmdb(path):
    return lmdb.open(str(path), subdir=False, readonly=True, lock=False,
                     readahead=False, meminit=False, max_readers=16)

def inspect_lmdb(path, sample_indices=(0,1,2)):
    env=open_lmdb(path)
    try:
        with env.begin() as txn:
            n=txn.stat()['entries']; samples=[]
            for i in sample_indices:
                if i >= n: continue
                raw=txn.get(str(i).encode('ascii'))
                if raw is None: raise ValueError(f'{path}: numeric index {i} missing')
                rec=pickle.loads(raw)
                samples.append({'index':i,'fields':sorted(map(str,rec.keys())),
                                'smiles':str(rec.get('smi'))[:90],
                                'conformations':len(rec.get('coordinates',[]))})
        return {'path':str(path),'record_count':n,'sample_records':samples}
    finally: env.close()

STRUCTURE={split:inspect_lmdb(DATA_DIR/f'{split}.lmdb') for split in ('train','valid')}
for split,v in STRUCTURE.items():
    print(f'[{split}] records={v["record_count"]:,}')
    display(v['sample_records'])
assert STRUCTURE['valid']['record_count'] >= N_EVAL


### B1 — Quick preview (never a substitute for full 18M audit)

The preview examines the first `PREVIEW_RECORDS_PER_SPLIT` records *in dataset numeric index order*. It is **not a random sample** and the observed duplicate fraction must **not** be extrapolated to the complete dataset. Use B2 for exact counts.


In [ ]:
from collections import Counter

def preview_split(path, nmax):
    env=open_lmdb(path)
    smi=[]; bad=0
    try:
        with env.begin() as txn:
            n=min(nmax,txn.stat()['entries'])
            for i in range(n):
                raw=txn.get(str(i).encode('ascii'))
                if raw is None: raise ValueError(f'Missing numeric LMDB key {i}')
                val=pickle.loads(raw).get('smi')
                if not isinstance(val,str) or not val.strip():bad+=1;continue
                smi.append(val)
    finally: env.close()
    return {'scanned':n,'valid_smiles':len(smi),'invalid_smiles':bad,
            'distinct_raw_smiles':len(set(smi)), 'repeated_records':len(smi)-len(set(smi)),
            'smiles':set(smi)}

if RUN_PREVIEW:
    a=preview_split(DATA_DIR/'train.lmdb',PREVIEW_RECORDS_PER_SPLIT)
    b=preview_split(DATA_DIR/'valid.lmdb',PREVIEW_RECORDS_PER_SPLIT)
    print('PREVIEW ONLY — not full dataset proportions')
    display({'train_scanned':a['scanned'],'train_repeated_records':a['repeated_records'],
             'valid_scanned':b['scanned'],'valid_repeated_records':b['repeated_records'],
             'preview_intersection':len(a['smiles']&b['smiles'])})
else: print('Preview disabled')


### B2 — Full exact audit (resumable, no Slurm)

The following uses **one on-disk SQLite index** (`identity TEXT PRIMARY KEY`, `flags` 1=train, 2=valid) and streams each LMDB in **numeric dataset order**, matching HEART's own reader. All LMDBs are opened read-only; SQLite is created **only under `OUTPUT_ROOT`**. Every chunk commits its cursor atomically with the inserts so interrupted scans resume without double-counting. The result is exact for the chosen identity definition, not an estimate. It can take hours and use multiple GB of scratch. SQLite database files should be retained until audit completion if interrupted.

If invalid/empty SMILES exist, they are counted separately and excluded from the identity denominator; such records prevent making a simple `record_count → unique` statement without qualification.


In [ ]:
import sqlite3

def canonical_key(s):
    from rdkit import Chem, RDLogger
    m=Chem.MolFromSmiles(s)
    if m is None: return None
    return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)

def audit_full(kind='raw', commit_every=25000):
    if kind not in ('raw','canonical'): raise ValueError(kind)
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    db_path=OUTPUT_ROOT/f'ligand_identity_{kind}.sqlite'
    con=sqlite3.connect(str(db_path), timeout=120)
    con.execute('PRAGMA journal_mode=WAL')
    con.execute('PRAGMA synchronous=NORMAL')
    con.execute('PRAGMA busy_timeout=120000')
    con.execute('CREATE TABLE IF NOT EXISTS ids (smi TEXT PRIMARY KEY, flags INTEGER NOT NULL) WITHOUT ROWID')
    con.execute('CREATE TABLE IF NOT EXISTS state (name TEXT PRIMARY KEY, val TEXT NOT NULL)')
    con.commit()
    def get(k,default='0'):
        row=con.execute('SELECT val FROM state WHERE name=?',(k,)).fetchone()
        return row[0] if row else default
    def update(k,v):con.execute('INSERT INTO state(name,val) VALUES(?,?) ON CONFLICT(name) DO UPDATE SET val=excluded.val',(k,str(v)))
    prior_kind=get('identity_kind',kind)
    if prior_kind!=kind: raise RuntimeError(f'Existing DB identity kind {prior_kind} != {kind}')
    con.execute('BEGIN'); update('identity_kind',kind);con.commit()
    rdkit_issues=0
    start_all=time.time()
    try:
        for split,flag in [('train',1),('valid',2)]:
            path=DATA_DIR/f'{split}.lmdb'
            env=open_lmdb(path)
            try:
                with env.begin() as txn: length=txn.stat()['entries']
                expected=get(f'{split}_source_n',str(length))
                if int(expected)!=length: raise RuntimeError(f'{split} source record count changed: {expected} -> {length}')
                start=int(get(f'{split}_cursor'))
                skipped=int(get(f'{split}_invalid'))
                print(f'{split}: records={length:,}, resume from {start:,}, invalid so far={skipped:,}',flush=True)
                while start<length:
                    stop=min(length,start+commit_every)
                    # Use a read transaction covering only one chunk.
                    with env.begin() as txn:
                        batch=[]
                        for idx in range(start,stop):
                            blob=txn.get(str(idx).encode('ascii'))
                            if blob is None: raise RuntimeError(f'{split}: missing numeric key {idx}')
                            val=pickle.loads(blob).get('smi')
                            if not isinstance(val,str) or not val.strip():skipped+=1;continue
                            key=val if kind=='raw' else canonical_key(val)
                            if key is None:skipped+=1;continue
                            batch.append((key,flag))
                    try:
                        con.execute('BEGIN')
                        con.executemany('INSERT INTO ids(smi,flags) VALUES(?,?) ON CONFLICT(smi) DO UPDATE SET flags = flags | excluded.flags',batch)
                        update(f'{split}_cursor',stop)
                        update(f'{split}_source_n',length)
                        update(f'{split}_invalid',skipped)
                        con.commit()
                    except BaseException:
                        con.rollback();raise
                    start=stop
                    if start==length or (start//commit_every)%20==0:
                        print(f'  {split}: {start:,}/{length:,} ({start/length:.1%}) | {time.time()-start_all:,.0f}s',flush=True)
            finally:env.close()
        total=con.execute('SELECT COUNT(*) FROM ids').fetchone()[0]
        train_unique,valid_unique,overlap=con.execute('SELECT SUM((flags & 1) != 0), SUM((flags & 2) != 0), SUM(flags = 3) FROM ids').fetchone()
        ntrain=int(get('train_source_n'));nvalid=int(get('valid_source_n'))
        itrain=int(get('train_invalid'));ivalid=int(get('valid_invalid'))
        result={'identity_kind':kind,'train_records':ntrain,'valid_records':nvalid,
                'train_invalid_records':itrain,'valid_invalid_records':ivalid,
                'train_distinct':int(train_unique),'valid_distinct':int(valid_unique),
                'train_repeated_records':ntrain-itrain-int(train_unique),
                'valid_repeated_records':nvalid-ivalid-int(valid_unique),
                'train_duplicate_rate_of_valid_smi_records':(ntrain-itrain-int(train_unique))/max(ntrain-itrain,1),
                'train_valid_distinct_overlap':int(overlap),
                'fraction_valid_distinct_seen_in_train':int(overlap)/max(int(valid_unique),1),
                'union_distinct':int(total),'lumi_reported_general_distinct':13369320,
                'train_distinct_minus_lumi':int(train_unique)-13369320,
                'db_file':str(db_path),'completed':True}
        out=OUTPUT_ROOT/f'ligand_identity_{kind}_summary.json'
        out.write_text(json.dumps(result,indent=2))
        display(result)
        return result
    finally: con.close()

AUDIT_RAW = audit_full('raw') if RUN_FULL_RAW_AUDIT else None
AUDIT_CANONICAL = audit_full('canonical') if RUN_FULL_CANONICAL_AUDIT else None
if not RUN_FULL_RAW_AUDIT:
    print('Exact raw audit not run. Set RUN_FULL_RAW_AUDIT=True in Section A, then run this cell.')


### B3 — Compare to LUMI-lab **without fitting** the number

A numerical match to 13.369M is not proof of matching chemistry: Uni-Mol versions, filtering and deduplication standards may differ. Conversely a mismatch does not invalidate training. Report both **record count** and **unique SMILES identities**, and clearly state the identity rule. If no full audit completed, comparison shows `NOT MEASURED` rather than an invented result.


In [ ]:
import pandas as pd
LUMI_13M=13_369_320
out=[]
for mode in ('raw','canonical'):
    j=OUTPUT_ROOT/f'ligand_identity_{mode}_summary.json'
    if j.exists():
        r=json.loads(j.read_text())
        if r.get('completed'):
            out.append({'Dataset':'HEART train','identity_definition':mode,'unique_molecules':r['train_distinct'],
                        'repeated_records':r['train_repeated_records'],
                        'relative_to_LUMI':r['train_distinct']-LUMI_13M})
out.append({'Dataset':'LUMI reported','identity_definition':'LUMI paper selection','unique_molecules':LUMI_13M,
            'repeated_records':None,'relative_to_LUMI':0})
df_comparison=pd.DataFrame(out)
display(df_comparison)
if len(out)==1:print('Full HEART duplicate rate is NOT YET MEASURED. Preview cannot establish 13M.')


## C — H200 checkpoint + original HEART objective evaluation

- Evaluates **10,000** molecules randomly sampled from the 98,844-record `valid.lmdb`, using `numpy.random.default_rng(17)` and **without replacement**.
- Uses the **original H200 Notebook's batching**: 8 molecules/batch, `dataset.set_epoch(1)`, `dataset.collater([dataset[i] ...])`, two independent view tensors with ordering `[view0 of 8, view1 of 8]`.
- Uses **`task.build_loss(args)`** with trained checkpoint for *exact HEART pretraining objective definitions*. In addition, extracts per-molecule metrics, not just loss logs.
- NOTE: the original H200 pretraining had `batch_size=448`. **The NT-Xent loss at batch=8 is not directly comparable with the training-validation loss computed at batch=448.** This notebook does not silently pretend equivalence. Same architecture/loss and 2-view layout, different contrastive negative pool.
- **CPU or GPU Jupyter kernel:** automatically detects the device; no new job. Running 10k on CPU may take a long time. Notebook is resumable via per-batch shards.
- **A non-collapsed representation / successful masked recovery cannot prove downstream lipid mRNA delivery prediction.** There are no biological delivery labels in this validation set.


In [ ]:
import torch, inspect, copy

for item in (str(MODEL_ROOT), str(UNICORE_ROOT)):
    if item not in sys.path:sys.path.insert(0,item)
from unicore import tasks as unicore_tasks
import unimol
assert Path(unimol.__file__).resolve().is_relative_to((MODEL_ROOT/'unimol').resolve()), unimol.__file__
assert 'heart_gpu' in str(sys.prefix), 'Switch Jupyter kernel to the trained HEART heart_gpu environment.'
import lmdb

torch.set_num_threads(TORCH_THREADS)
DEVICE=torch.device(('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE_PREFERENCE=='auto' else DEVICE_PREFERENCE)
if DEVICE.type=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA selected but not available')
print('Device:',DEVICE,'| torch:',torch.__version__,'| source:',unimol.__file__)
print('Checkpoint:',CHECKPOINT)
print('Batch method: 8 molecules -> 16 sequences, 2 contrastive views')


In [ ]:
# Load checkpoint only when starting the actual evaluation; trusted checkpoint written by our lab.
EVAL_CONTEXT=None

def get_eval_context():
    global EVAL_CONTEXT
    if EVAL_CONTEXT is not None:return EVAL_CONTEXT
    if not CHECKPOINT.exists():raise FileNotFoundError(CHECKPOINT)
    ck=torch.load(CHECKPOINT,map_location='cpu',weights_only=False,mmap=True)
    args=copy.deepcopy(ck['args'])
    args.data=str(DATA_DIR)
    args.num_workers=0
    args.mode='train'  # required to receive reconstruction heads and projection, not inference-only pair reps
    task=unicore_tasks.setup_task(args)
    model=task.build_model(args)
    model.load_state_dict(ck['model'],strict=True)
    model.to(DEVICE).eval()
    loss_fn=task.build_loss(args).to(DEVICE).eval()
    task.load_dataset('valid',combine=False)
    ds=task.dataset('valid');ds.set_epoch(VALID_EPOCH)
    print('Validation count:',len(ds),'| model parameters:',sum(t.numel() for t in model.parameters()))
    print('Loss class:',type(loss_fn).__name__,'| 2-view mode:',getattr(args,'contrastive_view_mode',None))
    assert len(ds)==STRUCTURE['valid']['record_count']
    EVAL_CONTEXT=(args,task,model,loss_fn,ds)
    del ck;gc.collect()
    return EVAL_CONTEXT

def move_batch(obj):
    if torch.is_tensor(obj):return obj.to(DEVICE,non_blocking=False)
    if isinstance(obj,dict):return {k:move_batch(v) for k,v in obj.items()}
    if isinstance(obj,list):return [move_batch(x) for x in obj]
    if isinstance(obj,tuple):return tuple(move_batch(x) for x in obj)
    return obj

rng=np.random.default_rng(SAMPLE_SEED)
SAMPLE_INDICES=rng.choice(STRUCTURE['valid']['record_count'],size=N_EVAL,replace=False).astype(np.int64)
print('Sample N:',len(SAMPLE_INDICES),'first eight:',SAMPLE_INDICES[:8].tolist())


### C1 — Batch-wise quantitative evaluation, robust to notebook interruption

Each completed batch is saved as a numbered **result shard** under `OUTPUT_ROOT`, with a fixed sample hash. Re-running the cell skips completed shards after checking the manifest. No training or modifications to the checkpoint. Output includes atom hit/count, coordinate and distance targets vs predictions, NT-Xent, positive/negative cosine, top-1 **in-batch** retrieval, plus CLS/projector vectors for the same 10,000 sampled molecules.

**Normalization details:** coordinate RMSE is reported in the model's centered coordinate system (translation removed, no biological position label). Distance recovery in the original HEART loss uses normalized distance targets; here actual distance RMSE is computed by undoing the original criterion's `dist_mean` and `dist_std` on the prediction. These are interpretable diagnostics, not a new loss function.


In [ ]:
import torch.nn.functional as F
from collections import defaultdict

EVAL_DIR=OUTPUT_ROOT/f'evaluation_n{N_EVAL}_seed{SAMPLE_SEED}_batch{BATCH_MOLECULES}'
SAMPLE_HASH=hashlib.sha256(SAMPLE_INDICES.tobytes()).hexdigest()

def safe_float(x):
    if isinstance(x,torch.Tensor):return float(x.detach().cpu().item()) if x.numel()==1 else None
    return float(x) if isinstance(x,(float,int,np.number)) else None

def eval_batch(batch_indices,args,model,loss_fn,ds):
    n=len(batch_indices)
    sample=ds.collater([ds[int(i)] for i in batch_indices])
    names=list(sample['target']['smi_name'])
    assert len(names)==2*n and names[:n]==names[n:], 'Unexpected contrastive view ordering'
    src=sample['net_input']['src_tokens']
    assert src.shape[0]==2*n
    sample=move_batch(sample)
    # Hook only the existing encoder and the model's original six-field output;
    # criterion(model, sample) invokes a SINGLE trained forward, unchanged.
    captured={}
    def encoder_hook(mod,ins,outs):captured['cls']=outs[0][:,0,:].detach().float().cpu()
    def model_hook(mod,ins,outs):
        captured['dist']=outs[1].detach().float().cpu() if outs[1] is not None else None
        captured['coord']=outs[2].detach().float().cpu() if outs[2] is not None else None
        captured['projector']=outs[5].detach().float().cpu()
    h1=model.encoder.register_forward_hook(encoder_hook)
    h2=model.register_forward_hook(model_hook)
    try:
        with torch.inference_mode():
            total_loss,sz,log=loss_fn(model,sample)
    finally:h1.remove();h2.remove()
    sample_cpu={k:{name:v.detach().cpu() if torch.is_tensor(v) else v for name,v in sd.items()} for k,sd in sample.items()}
    targets=sample_cpu['target']; inputs=sample_cpu['net_input']
    mask=targets['tokens_target'].ne(loss_fn.padding_idx)
    token_count=int(mask.sum())
    token_hits=int(safe_float(log['masked_token_hit']))
    raw={k:safe_float(v) for k,v in log.items()}
    raw['criterion_returned_loss']=safe_float(total_loss)
    raw['batch_molecules']=n
    # Geometry per masked atom: one coordinate vector per target, as in original loss.
    coord=captured['coord']; dist=captured['dist']
    if coord is not None:
        c0=inputs['src_coord'][mask].float(); c1=targets['coord_target'][mask].float(); c2=coord[mask].float()
        raw['coordinate_prediction_squared_error_sum']=float(((c2-c1)**2).sum())
        raw['coordinate_noisy_baseline_squared_error_sum']=float(((c0-c1)**2).sum())
        raw['coordinate_components']=int(c1.numel())
    if dist is not None:
        dtrue=targets['distance_target'].float()
        pred=dist.float()*float(loss_fn.dist_std)+float(loss_fn.dist_mean)
        # Original distance head: each masked-atom row to all non-padding positions.
        keep=mask.unsqueeze(-1) & inputs['src_tokens'].ne(loss_fn.padding_idx).unsqueeze(1)
        raw['distance_prediction_squared_error_sum']=float(((pred-dtrue)[keep]**2).sum())
        raw['distance_pairs']=int(keep.sum())
    p=captured['projector'].numpy().astype('float32')
    u=p[:n];v=p[n:]
    # In-batch retrieval ONLY (not 10k-way accuracy); positive pairs same molecule.
    sim=u@v.T
    raw['contrastive_pos_similarity_sum']=float(np.trace(sim))
    raw['contrastive_neg_similarity_sum']=float(sim.sum()-np.trace(sim))
    raw['contrastive_neg_count']=n*(n-1)
    raw['contrastive_matched_top1']=int((sim.argmax(axis=1)==np.arange(n)).sum())
    raw['contrastive_pair_count']=n
    metadata=[]
    lengths=inputs['src_tokens'][:n].ne(loss_fn.padding_idx).sum(dim=1)
    for j,idx in enumerate(batch_indices):
        metadata.append({'dataset_index':int(idx),'smi':str(names[j]),
                         'atom_tokens':int(lengths[j])-2,
                         'masked_atoms_view0':int(mask[j].sum()),'masked_atoms_view1':int(mask[n+j].sum())})
    return {'logs':raw,'metadata':metadata,
            'cls_view0':captured['cls'][:n].numpy().astype('float32'),
            'cls_view1':captured['cls'][n:].numpy().astype('float32'),
            'projector_view0':p[:n], 'projector_view1':p[n:]}

def execute_eval(limit=None):
    # Only call this explicitly, not automatically when opening the notebook.
    args,task,model,loss_fn,ds=get_eval_context()
    EVAL_DIR.mkdir(parents=True,exist_ok=True)
    manifest_path=EVAL_DIR/'manifest.json'
    manifest={'sample_hash':SAMPLE_HASH,'n_requested':N_EVAL,'batch_molecules':BATCH_MOLECULES,
              'seed':SAMPLE_SEED,'epoch':VALID_EPOCH,'checkpoint':str(CHECKPOINT),
              'checkpoint_size':CHECKPOINT.stat().st_size,'checkpoint_mtime_ns':CHECKPOINT.stat().st_mtime_ns,
              'data_valid_records':len(ds)}
    if manifest_path.exists():
        if json.loads(manifest_path.read_text())!=manifest:
            raise RuntimeError('Existing evaluation manifest does not match checkpoint/sample/config! Use a new output folder.')
    else:manifest_path.write_text(json.dumps(manifest,indent=2))
    end=N_EVAL if limit is None else min(int(limit),N_EVAL)
    start_time=time.time()
    for start in range(0,end,BATCH_MOLECULES):
        idx=SAMPLE_INDICES[start:min(start+BATCH_MOLECULES,N_EVAL)]
        shard=EVAL_DIR/f'batch_{start:06d}.npz'
        if shard.exists():continue
        result=eval_batch(idx,args,model,loss_fn,ds)
        tmp=EVAL_DIR/f'batch_{start:06d}.npz.tmp'
        with tmp.open('wb') as f:
            np.savez_compressed(f,
                 logs_json=json.dumps(result['logs']),
                 metadata_json=json.dumps(result['metadata']),
                 cls_view0=result['cls_view0'],cls_view1=result['cls_view1'],
                 projector_view0=result['projector_view0'],projector_view1=result['projector_view1'])
        os.replace(tmp,shard)
        if start==0 or (start//BATCH_MOLECULES+1)%50==0:
            print(f'{min(start+len(idx),N_EVAL):,}/{N_EVAL:,} molecules | {time.time()-start_time:.0f}s this session',flush=True)
    print('Batch shards complete/present:',len(list(EVAL_DIR.glob('batch_*.npz'))),'expected',math.ceil(N_EVAL/BATCH_MOLECULES))

# 8-molecule smoke can be run without turning on the full evaluation:
RUN_SMOKE=False
if RUN_SMOKE:
    execute_eval(limit=8)
if RUN_EVALUATION:
    execute_eval()
else:
    print('10k forward evaluation NOT started. Set RUN_EVALUATION=True (and select heart_gpu kernel).')


## D — Summarize the original pretraining objectives and representations

This section reads saved batch shards only. It will not run a model or submit jobs. Incomplete results are labeled as such; plots use currently available sampled molecules, but no full-10k claim until all expected batches exist.


In [ ]:
import pandas as pd
import numpy as np

def read_shards():
    shards=sorted(EVAL_DIR.glob('batch_*.npz')) if EVAL_DIR.exists() else []
    logs=[]; rows=[]; arrays=defaultdict(list)
    for path in shards:
        with np.load(path,allow_pickle=False) as f:
            logs.append(json.loads(str(f['logs_json'])))
            rows.extend(json.loads(str(f['metadata_json'])))
            for k in ['cls_view0','cls_view1','projector_view0','projector_view1']:
                arrays[k].append(f[k].copy())
    emb={k:np.concatenate(v,axis=0) for k,v in arrays.items() if v}
    return logs,pd.DataFrame(rows),emb

BATCH_LOGS,MOL_META,EMBEDDINGS=read_shards()
COMPLETE=(len(BATCH_LOGS)==math.ceil(N_EVAL/BATCH_MOLECULES) and len(MOL_META)==N_EVAL)
print('Available batches:',len(BATCH_LOGS),'/',math.ceil(N_EVAL/BATCH_MOLECULES),
      '| molecules:',len(MOL_META),'/',N_EVAL,'| FULL EVALUATION COMPLETE:',COMPLETE)
if BATCH_LOGS:
    def s(k):return sum((r.get(k) or 0) for r in BATCH_LOGS)
    report={
        'molecules_evaluated':int(len(MOL_META)),
        'masked_atom_accuracy':s('masked_token_hit')/max(s('masked_token_cnt'),1),
        'masked_token_loss_mean_of_batch_losses':s('masked_token_loss')/len(BATCH_LOGS),
        'masked_coord_loss_mean_of_batch_losses':s('masked_coord_loss')/len(BATCH_LOGS),
        'masked_distance_loss_mean_of_batch_losses':s('masked_dist_loss')/len(BATCH_LOGS),
        'contrastive_NT_Xent_mean_of_batch_losses':s('contrastive_loss')/len(BATCH_LOGS),
        'total_weighted_HEART_loss_mean_of_batches':s('loss')/len(BATCH_LOGS),
        'coordinate_RMSE':math.sqrt(s('coordinate_prediction_squared_error_sum')/max(s('coordinate_components'),1)),
        'noisy_input_coordinate_RMSE':math.sqrt(s('coordinate_noisy_baseline_squared_error_sum')/max(s('coordinate_components'),1)),
        'distance_RMSE_angstroms':math.sqrt(s('distance_prediction_squared_error_sum')/max(s('distance_pairs'),1)),
        'contrastive_positive_cosine_mean':s('contrastive_pos_similarity_sum')/max(s('contrastive_pair_count'),1),
        'contrastive_negative_cosine_mean':s('contrastive_neg_similarity_sum')/max(s('contrastive_neg_count'),1),
        'contrastive_top1_in_batch':s('contrastive_matched_top1')/max(s('contrastive_pair_count'),1),
        'contrastive_chance_in_batch':1/BATCH_MOLECULES,
        'full_10000_complete':COMPLETE,
    }
    display(pd.DataFrame([{'metric':k,'value':v} for k,v in report.items()]))
    if COMPLETE:
        EVAL_DIR.joinpath('evaluation_objectives_summary.json').write_text(json.dumps(report,indent=2))
else: print('No H200 inference results yet; training log metrics are historical evidence, not this notebook evaluation.')


## E — PCA / UMAP / embedding quality (10,000 molecular representations)

The raw CLS is the output of the final Transformer encoder at the `<s>` token. The projected embedding is the normalized contrastive vector used in NT-Xent. All embedding rows keep exact corresponding dataset index + SMILES; no LMDB cursor ordering ambiguity.

**Scaling consideration:** the 192-molecule notebook calculated full `N×N` cosine matrices and full-SVD. For 10,000 molecules, avoid an unneeded 100M-pair matrix: sample pair similarities and run randomized PCA. UMAP retains cosine distance. Saving output is read-only relative to training data/checkpoint.


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

if not EMBEDDINGS:
    print('No embeddings available. Run Section C1 first.')
else:
    keys=list(EMBEDDINGS)
    emb_rows=[]
    for k,X in EMBEDDINGS.items():
        norms=np.linalg.norm(X,axis=1)
        var=X.var(axis=0)
        rng=np.random.default_rng(17)
        n=len(X); pairs=min(50000,max(n*10,1))
        i=rng.integers(0,n,size=pairs);j=rng.integers(0,n,size=pairs)
        ok=i!=j
        x0=X[i[ok]];x1=X[j[ok]]
        cos=np.sum(x0*x1,axis=1)/(np.maximum(np.linalg.norm(x0,axis=1)*np.linalg.norm(x1,axis=1),1e-12))
        emb_rows.append({'representation':k,'n':n,'dimension':X.shape[1],
                         'nan_or_inf':int((~np.isfinite(X)).sum()),
                         'zero_variance_features':int((var<1e-12).sum()),
                         'mean_embedding_norm':float(norms.mean()),
                         'random_pairs_cosine_mean':float(np.mean(cos)),
                         'random_pairs_cosine_std':float(np.std(cos))})
    display(pd.DataFrame(emb_rows))


In [ ]:
if EMBEDDINGS and len(MOL_META)>=20:
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt
    X=EMBEDDINGS['cls_view0']
    pca=PCA(n_components=min(20,len(X)-1,X.shape[1]),svd_solver='randomized',random_state=SAMPLE_SEED)
    xy=pca.fit_transform(X)
    atoms=MOL_META['atom_tokens'].to_numpy()
    fig,ax=plt.subplots(figsize=(7,5))
    scatter=ax.scatter(xy[:,0],xy[:,1],c=atoms,s=4 if len(X)>1000 else 16,alpha=.75)
    ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
           title=f'HEART H200 | raw CLS | PCA | n={len(X):,}')
    fig.colorbar(scatter,ax=ax,label='atom count (model input)');plt.show()
    if COMPLETE:
        np.save(EVAL_DIR/'embeddings_cls_10k.npy',X)
        MOL_META.to_csv(EVAL_DIR/'molecule_metadata_10k.csv',index=False)
        pd.DataFrame({'dataset_index':MOL_META['dataset_index'],'PC1':xy[:,0],'PC2':xy[:,1]}).to_csv(EVAL_DIR/'pca_10k.csv',index=False)
else:print('Need >=20 completed molecule embeddings for PCA.')


In [ ]:
if EMBEDDINGS and len(MOL_META)>=20:
    try:
        import umap
        um=umap.UMAP(n_neighbors=15,min_dist=.1,metric='cosine',random_state=SAMPLE_SEED)
        xy_umap=um.fit_transform(EMBEDDINGS['cls_view0'])
        fig,ax=plt.subplots(figsize=(7,5))
        scatter=ax.scatter(xy_umap[:,0],xy_umap[:,1],c=MOL_META['atom_tokens'],s=4 if len(MOL_META)>1000 else 16,alpha=.7)
        ax.set(xlabel='UMAP1',ylabel='UMAP2',title=f'HEART H200 | raw CLS | UMAP | n={len(MOL_META):,}')
        fig.colorbar(scatter,ax=ax,label='atom count (model input)');plt.show()
        if COMPLETE:
            pd.DataFrame({'dataset_index':MOL_META['dataset_index'],'UMAP1':xy_umap[:,0],'UMAP2':xy_umap[:,1]}).to_csv(EVAL_DIR/'umap_10k.csv',index=False)
    except ImportError:
        print('umap-learn unavailable in current kernel; UMAP skipped. No packages installed.')
else:print('Need >=20 molecule embeddings for UMAP.')


## F — Evidence-based verdict to report to PI

**Dataset:** Report *exact* record count, distinct raw and/or canonical SMILES, duplicate rate, and validation overlap. Only after exact enumeration should you compare numbers with LUMI's 13,369,320. Do **not** write “18.84M deduped to 13M” without confirming it.

**Training:** Original decreasing train/validation loss and a high masked-atom score are encouraging but not proof of zero generalization error. The masked-atom accuracy from this notebook is computed as `masked_token_hit / masked_token_cnt`, exactly matching HEART's source criterion.

**Geometry:** Compare recovered coordinate RMSE with the original noisy-input RMSE. A lower model RMSE indicates the network learned denoising on this held-out validation sample. Distance RMSE in Å is a supplemental metric; HEART's own `masked_dist_loss` uses normalized targets and Smooth L1.

**Contrastive:** Compare matched-view cosine vs unmatched-view cosine, and in-batch top-1 retrieval vs **1/8 chance**. The batch-of-8 NT-Xent loss is a legitimate objective diagnostic but **not numerically comparable to original training validation with batch 448**.

**Representation:** Non-collapsed CLS vectors, diversity in PCA and chemistry-meaningful similarity support learned representations; PCA/UMAP **alone** do not establish successful molecule-property prediction, and certainly not mRNA delivery performance. To make a downstream prediction claim, one needs labeled property or delivery data.

**Data leakage:** If the complete audit finds many train–valid shared identities, historical validation scores can be optimistic. A clean molecular-disjoint held-out set would be required for stronger claims. Do not discard the checkpoint solely because the record counts differ from LUMI.

**Execution safety:** Notebook does not submit SLURM jobs or modify the source LMDB/checkpoints. It does write SQLite audit state and per-batch evaluation artifacts under `OUTPUT_ROOT`.
